In [0]:
# Demo 12 Setup: Metric Views
# Creates a Gold-layer orders table suitable for defining a metric view.

import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "main"
schema = f"demo_metric_views_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Schema: {catalog}.{schema}")

In [0]:
# --- gold_orders: 1500 rows ---
# A Gold-layer table with pre-joined dimension attributes.
# This is the kind of table you'd build a metric view on top of.

random.seed(77)

spark.sql("DROP TABLE IF EXISTS gold_orders")

regions = ['Northeast', 'Southeast', 'Midwest', 'West']
categories = ['Electronics', 'Clothing', 'Home', 'Sports', 'Food']
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
channels = ['online', 'store', 'partner']

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("region", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("customer_segment", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("revenue", DoubleType(), False),
    StructField("cost", DoubleType(), False)
])

rows = []
for i in range(1, 1501):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    revenue = round(random.uniform(25, 800), 2)
    rows.append((
        i,
        order_date,
        random.randint(1, 100),
        random.choice(regions),
        random.choice(categories),
        random.choice(segments),
        random.choice(channels),
        random.randint(1, 8),
        revenue,
        round(revenue * random.uniform(0.4, 0.75), 2)
    ))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("gold_orders")
print(f"Created gold_orders: {df.count()} rows")

In [0]:
# --- Summary ---
print("="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Table created:")
print(f"  gold_orders - 1500 rows")
print(f"    Dimensions: region, product_category, customer_segment, channel, order_date")
print(f"    Measures:   quantity, revenue, cost")
print(f"")
print(f"Next: Open the SQL Editor and run:")
print(f"  USE CATALOG {catalog};")
print(f"  USE SCHEMA {schema};")
print(f"")
print(f"Then follow the Demo 12 notebook to create and query a metric view.")